 Automatically classify waste products using image recognition techniques. <br>
 This project aims to build a model that can differentiate between recyclable and organic waste products using transfer learning

In [1]:
import importlib.util
import sys

packages = {
    "numpy": "numpy==2.2.0",
    "pandas": "pandas==2.2.3",
    "sklearn": "scikit-learn==1.6.0",
    "matplotlib": "matplotlib==3.9.3",
    "tensorflow": "tensorflow==2.18.0",
    "seaborn": "seaborn==0.13.2",
    "pyarrow": "pyarrow",
    "requests": "requests",
}

for module_name, pip_name in packages.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing {pip_name} ...")
        !{sys.executable} -m pip install {pip_name}
    else:
        print(f"{module_name} is already installed")

numpy is already installed
pandas is already installed
sklearn is already installed
matplotlib is already installed
tensorflow is already installed
seaborn is already installed
pyarrow is already installed
requests is already installed


Importing required libraries

In [2]:
import numpy as np
import os
# import random, shutil
import glob


from matplotlib import pyplot as plt
from matplotlib import pyplot
from matplotlib.image import imread

# from os import makedirs,listdir
# from shutil import copyfile
# from random import seed
# from random import random

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras import optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
# from tensorflow.keras.layers import Conv2D, MaxPooling2D,GlobalAveragePooling2D, Input
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# from tensorflow.keras.applications import InceptionV3
from sklearn import metrics

import warnings
warnings.filterwarnings('ignore')


In [4]:
print(tf.__version__)

2.18.0


I train an algorithm on images and to predict the labels for images in my test set (1 = recyclable, 0 = organic)

In [5]:
import os
import zipfile
from tqdm import tqdm

# Exact zip file name in your folder
file_name = "fruits-360-original-size.zip"

# Check that the zip file exists
if not os.path.exists(file_name):
    raise FileNotFoundError(f"{file_name} not found. Make sure it is in the same folder as your notebook.")

def extract_file_with_progress(file_name):
    print("Extracting file with progress...")

    with zipfile.ZipFile(file_name, "r") as zip_ref:
        members = zip_ref.infolist()

        with tqdm(total=len(members), unit="file") as progress_bar:
            for member in members:
                zip_ref.extract(member)
                progress_bar.update(1)

    print("Finished extracting file.")

extract_file_with_progress(file_name)

# Show extracted folders/files
print("\nFiles/folders in current directory:")
print(os.listdir())

Extracting file with progress...


100%|██████████| 12583/12583 [00:35<00:00, 356.99file/s]

Finished extracting file.

Files/folders in current directory:
['.git', '.gitignore', 'fruits-360-original-size', 'fruits-360-original-size.zip', 'Keras-TrashClassification-TransferLearning.ipynb', 'README.md']


Model configuration options.

In [6]:
img_rows, img_cols = 150, 150
batch_size = 32
n_epochs = 10
n_classes = 2
val_split = 0.2
verbosity = 1
path = 'o-vs-r-split/train/'
path_test = 'o-vs-r-split/test/'
input_shape = (img_rows, img_cols, 3)
labels = ['O', 'R']
seed = 42